In [ ]:
import glob
import os
import numpy as np
import pandas as pd

from functions import *

#### 1. Load data and basic demographics

In [ ]:
cpet_files = sorted(glob.glob('data/CPET/*.csv'))
nirs_files = sorted(glob.glob('data/NIRS/*.csv'))
cpet_basenames = [os.path.splitext(os.path.basename(f))[0] for f in cpet_files]
nirs_basenames = [os.path.splitext(os.path.basename(f))[0] for f in nirs_files]
assert len({b.split('_')[0] for b in cpet_basenames}) == len(cpet_basenames), 'expected one CPET file per participant'
print(f"All filenames match between CPET and NIRS for analysis: {set(cpet_basenames) == set(nirs_basenames)}")
TBT_CPET_NIRS_for_analysis = sorted(set(cpet_basenames) & set(nirs_basenames))
print("n =", len(TBT_CPET_NIRS_for_analysis))

In [ ]:
participant_info = pd.read_csv('data/metadata/participant_info.csv')
participant_info['nmd_type'] = participant_info['clinical_diagnosis'].map(NMD_MAPPING)
id_to_diag = dict(zip(participant_info['participant_id'], participant_info['nmd_type']))

demographics = pd.read_csv('data/metadata/cpet_demographics.csv')
demographics['height'] = demographics['height'] / 100
slope_df = pd.read_csv('data/metadata/cpet_slopes.csv')
pt_pro_df = pd.read_csv('data/metadata/PRO_PT_redcap.csv')
keiser_watt_df = pd.read_csv('data/metadata/cpet_watts_borg.csv')[['participant_id'] + WATT_COLS + BORG_COLS]
for name, df_ in {'participant_info': participant_info, 'demographics': demographics, 'slope_df': slope_df,
                  'pt_pro_df': pt_pro_df, 'keiser_watt_df': keiser_watt_df}.items():
    assert df_['participant_id'].is_unique, f'{name}: expected one row per participant'

cpet_info_df = pd.DataFrame({'filename': cpet_basenames})
cpet_info_df[['participant_id', 'testtype']] = cpet_info_df['filename'].str.split('_', expand=True)
cpet_demographics = cpet_info_df.merge(demographics, on=['participant_id', 'testtype'], how='left', validate='1:1')
assert cpet_demographics['height'].notna().all(), 'some CPET files have no matching demographics row'
cpet_demographics['bmi'] = cpet_demographics['weight'] / (cpet_demographics['height'] ** 2)
cpet_demographics = cpet_demographics.merge(participant_info[['participant_id', 'clinical_diagnosis', 'nmd_type']], on='participant_id', how='left', validate='m:1')
cpet_demographics['sex'] = cpet_demographics['sex'].map({'M': 'Male', 'F': 'Female'})
cpet_demographics.head()

In [ ]:
demo_info = get_patient_info(cpet_demographics)
demo_info.to_csv('outputs/tables/patient_info.csv')
demo_info

In [ ]:
demo_info_by_diag = get_patient_info_by_diagnosis(cpet_demographics)
demo_info_by_diag.to_csv('outputs/tables/patient_info_by_diagnosis.csv')
demo_info_by_diag

#### 2. Create long form dataframe

In [ ]:
long_form_df = build_cpet_data(cpet_files, cpet_demographics, slope_df, keiser_watt_df)
long_form_df.head()

#### 3. Extract CPET peak and recovery metrics per diagnosis

In [ ]:
x_sectional_peak_table = make_peak_cpet_table(long_form_df, fdr_correction=True)
x_sectional_peak_table.to_csv('outputs/tables/cross_sectional_peak_table.csv')
x_sectional_peak_table

In [ ]:
missingness_report = {} # for CPET peak table footnote
for col in ['VT2_VO2_AT', 'VE_VCO2_slope', 'VT1_VE_VCO2_slope', 'VT2_VE_VCO2_slope']:
    for nmd_type in long_form_df['nmd_type'].unique():
        if nmd_type not in missingness_report:
            missingness_report[nmd_type] = {}
        group_df = long_form_df[long_form_df['nmd_type'] == nmd_type]
        total = len(group_df)
        n_missing = group_df[col].isna().sum()
        available = total - n_missing
        missingness_report[nmd_type][col] = f"{available}/{total}"
missingness_report_df = pd.DataFrame.from_dict(missingness_report, orient='index')
missingness_report_df

In [ ]:
x_sectional_recovery_table = make_recovery_table(long_form_df, fdr_correction=True)
x_sectional_recovery_table.to_csv('outputs/tables/cross_sectional_recovery_table.csv')
x_sectional_recovery_table

#### 4. Plot CPET recovery trajectories

In [ ]:
halftime_trajectories = graph_recovery_averages(cpet_files, cpet_demographics, 
                                                metrics=['VO2', 'HR', 'VE', 'VCO2', 'VO2/HR'],
                                                ylabels={'VO2': 'VO2 (mL/min)', 'HR': 'HR (bpm)', 'VE': 'VE (L/min)', 'VCO2': 'VCO2 (L/min)', 'VO2/HR': 'O2 pulse (mL/min)'},
                                                group_map=NMD_MAPPING, graph_smooth_sec=30, anchor_mode='exercise_end', ncols=2)

halftime_trajectories.savefig('outputs/trajectory_plots/cpet_halftime_trajectories.pdf', bbox_inches='tight')
halftime_trajectories

In [ ]:
overshoot_trajectories = graph_recovery_averages(cpet_files, cpet_demographics, 
                                                 metrics=['RQ', 'VE/VO2', 'VE/VCO2', 'PetO2'], 
                                                 ylabels={'RQ': 'RER', 'VE/VO2': 'VE/VO2 (L/min)', 'VE/VCO2': 'VE/VCO2 (L/min)', 'PetO2': 'PETO2 (mmHg)'},
                                                 group_map=NMD_MAPPING, graph_smooth_sec=30, anchor_mode='exercise_end', ncols=2, legend=False)
overshoot_trajectories.savefig('outputs/trajectory_plots/cpet_overshoot_trajectories.pdf', bbox_inches='tight')
overshoot_trajectories

#### 5. Plot CPET peak and recovery metrics vs. PROs

In [ ]:
plot_cpet_vs_pros(long_form_df, pt_pro_df, 'Structural', recovery_or_peak='recovery', fdr_correction=False, save_dir='outputs/corr_matrices')
plot_cpet_vs_pros(long_form_df, pt_pro_df, 'Non-structural', recovery_or_peak='recovery', fdr_correction=False, save_dir='outputs/corr_matrices')

In [ ]:
plot_cpet_vs_pros(long_form_df, pt_pro_df, 'Structural', recovery_or_peak='peak', fdr_correction=True, save_dir='outputs/corr_matrices')
plot_cpet_vs_pros(long_form_df, pt_pro_df, 'Non-structural', recovery_or_peak='peak', fdr_correction=True, save_dir='outputs/corr_matrices')

#### 6. Reporting NIRS exclusions for Supplement

In [ ]:
optodes_qc_df = pd.read_csv('data/metadata/nirs_optode_selection.csv', index_col='basename')
qc_df = optodes_qc_df.reset_index().merge(cpet_demographics[['filename', 'bmi']], left_on='basename', right_on='filename', how='left').drop(columns=['filename'])
qc_df['diagnosis'] = qc_df['diagnosis'].map(NMD_MAPPING)

n_excluded = sum((qc_df[cols] == False).all(axis=1).astype(int) for cols in OPTODE_COLS.values())
all_optode_cols = [c for cols in OPTODE_COLS.values() for c in cols]
in_analysis = lambda d: d[d['basename'].isin(TBT_CPET_NIRS_for_analysis)].sort_values('basename').reset_index(drop=True)

excluded_at_least_one_muscle = in_analysis(qc_df[n_excluded.between(1, 2)])
included_at_least_one_muscle = in_analysis(qc_df[(qc_df[all_optode_cols] == True).any(axis=1)])

excluded_at_least_one_muscle # descriptive exclusion table for supplement
excluded_at_least_one_muscle.to_csv("outputs/tables/excluded_at_least_one_muscle.csv", index=False)

In [ ]:
optode_select_tbl = build_optode_selection_supp_table(included_at_least_one_muscle)
optode_select_tbl.to_csv("outputs/tables/optode_selection_supplementary.csv", index=True)
optode_select_tbl

#### 7. Build all NIRS dataframes & CPET-NIRS merged dataframes

In [ ]:
nirs_dfs = load_nirs_dfs([f for f in nirs_files], id_to_diag, optodes_qc_df)
cpet_nirs_dfs = build_cpet_nirs_dfs(cpet_files, nirs_files, optodes_qc_df)

#### 8. Plot all NIRS trajectories

In [ ]:
nirs_trajectories = plot_TBT_trajectories_qc(nirs_dfs, optodes_qc_df)
nirs_trajectories.savefig('outputs/trajectory_plots/nirs_trajectories.pdf', dpi=600)
nirs_trajectories.savefig('img/nirs_trajectories.png', dpi=600)
nirs_trajectories

#### 9. Extract NIRS metrics per diagnosis

In [ ]:
tsi_df = build_TSI_summary_df(nirs_dfs)
tsi_long = tsi_df.melt(id_vars=['participant_id', 'diagnosis', 'muscle'],
                       value_vars=['tsi_recoveryslope'],
                       var_name='metric', value_name='value')
hb_long = build_delta_nirs_table(cpet_nirs_dfs, id_to_diag)[['participant_id', 'diagnosis', 'muscle', 'metric', 'value']]
nirs_long = pd.concat([hb_long, tsi_long], ignore_index=True)
mismatch = set(hb_long.participant_id) ^ set(tsi_long.participant_id)
assert not mismatch, f"mismatch between HB and TSI participant IDs: {sorted(mismatch)}"

nirs_table = create_nirs_combined_table(nirs_long)
nirs_table.to_csv('outputs/nirs_table4.csv', index=False)
nirs_table